# Player Tracking & Re-Identification Pipeline

End-to-end notebook: detection -> short-term tracking -> team clustering -> re-id recovery -> jersey OCR -> final annotated video.

**Run this on Google Colab** (Runtime -> Change runtime type -> T4 GPU) for the easiest setup, or on your own machine with Jupyter -- see the markdown notes at the bottom of this notebook for local setup instructions.

Run cells top to bottom in order. Each stage prints/shows something so you can sanity-check before moving to the next.

## Step 0 - Install dependencies

In [20]:
!pip install ultralytics supervision transformers scikit-learn -q
# torchreid is optional (Step 4 appearance embeddings) - install separately if the next cell errors:
# !pip install torchreid 

In [21]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO

print("GPU available:", torch.cuda.is_available())

GPU available: False


## Step 0b - Upload your video

On Colab, run the cell below and pick your file. On local Jupyter, just put the video in the same folder and set `VIDEO_PATH` directly.

In [22]:
try:
    from google.colab import files
    uploaded = files.upload()
    VIDEO_PATH = list(uploaded.keys())[0]
except ImportError:
    # local jupyter: edit this path to point at your own clip
    VIDEO_PATH = "clip.mp4.mp4"

print("Using video:", VIDEO_PATH)

Using video: clip.mp4.mp4


In [23]:
import subprocess
subprocess.run([
    "ffmpeg", "-y", "-i", VIDEO_PATH, "-t", "15",
    "-c:v", "libx264", "-preset", "veryfast", "-an", "short_clip.mp4"
])
VIDEO_PATH = "short_clip.mp4"
print("Now using:", VIDEO_PATH)

ffmpeg version 8.1.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.4.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1.2 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.102 / 60. 26.102
  libavcodec     62. 28.102 / 62. 28.102
  libavformat    62. 12.102 / 62. 12.102
  libavdevice    62.  3.102 / 62.  3.102
  libavfilter    11. 14.102 / 11. 14.102
  libswscale      9.  5.102 /  9.  5.102
  libswresample   6.  3.102 /  6.  3.102
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'clip.mp4.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso2mp41
  Duration: 00:07:02.19, start:

Now using: short_clip.mp4


[out#0/mp4 @ 0x12ef0e7c0] video:27358KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.042984%
frame=  900 fps= 93 q=-1.0 Lsize=   27370KiB time=00:00:14.96 bitrate=14980.9kbits/s speed=1.54x elapsed=0:00:09.71    
[libx264 @ 0x12ef06190] frame I:6     Avg QP:18.52  size:146156
[libx264 @ 0x12ef06190] frame P:226   Avg QP:20.70  size: 77528
[libx264 @ 0x12ef06190] frame B:668   Avg QP:21.36  size: 14395
[libx264 @ 0x12ef06190] consecutive B-frames:  0.9%  0.2%  0.7% 98.2%
[libx264 @ 0x12ef06190] mb I  I16..4: 45.8% 47.3%  6.9%
[libx264 @ 0x12ef06190] mb P  I16..4: 32.3% 20.4%  0.6%  P16..4: 25.8%  5.8%  1.7%  0.0%  0.0%    skip:13.5%
[libx264 @ 0x12ef06190] mb B  I16..4:  2.1%  0.8%  0.0%  B16..8:  9.5%  0.9%  0.0%  direct:11.6%  skip:75.0%  L0:60.6% L1:37.3% BI: 2.2%
[libx264 @ 0x12ef06190] 8x8 transform intra:37.4% inter:22.9%
[libx264 @ 0x12ef06190] coded y,uvDC,uvAC intra: 16.6% 31.0% 0.7% inter: 1.1% 19.7% 0.0%
[libx264 @ 0x12ef06190] i16 v,h,d

## Step 1 - Player detection (YOLOv8)

Quick sanity check on a single frame before running the full video.

In [24]:
det_model = YOLO("yolov8n.pt")  # auto-downloads ~6MB on first run

cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()

result = det_model.predict(frame, classes=[0], conf=0.25, verbose=False)[0]
annotated = result.plot()
cv2.imwrite("check_detection.jpg", annotated)
print("boxes found:", len(result.boxes))
print("saved check_detection.jpg -- open it and confirm boxes land on players")

boxes found: 10
saved check_detection.jpg -- open it and confirm boxes land on players


## Step 2 - Short-term tracking (BoT-SORT)

Same model, `.track()` instead of `.predict()`. This adds persistent IDs across frames using a Kalman motion model + camera-motion compensation.

In [25]:
track_results = det_model.track(
    source=VIDEO_PATH,
    classes=[0],
    conf=0.25,
    tracker="botsort.yaml",   # swap to "bytetrack.yaml" if this is too slow on CPU
    persist=True,
    stream=True,
    verbose=False,
)

# materialize into a list of per-frame track data so later steps can reuse it
all_frames = []          # list of raw BGR frames
all_tracks = []          # list of lists: [(track_id, x1, y1, x2, y2), ...] per frame
raw_id_set = set()

for r in track_results:
    frame = r.orig_img
    frame_tracks = []
    if r.boxes is not None and r.boxes.id is not None:
        ids = r.boxes.id.int().tolist()
        xyxy = r.boxes.xyxy.tolist()
        for tid, (x1, y1, x2, y2) in zip(ids, xyxy):
            frame_tracks.append((tid, x1, y1, x2, y2))
            raw_id_set.add(tid)
    all_frames.append(frame)
    all_tracks.append(frame_tracks)

print(f"frames processed: {len(all_frames)}")
print(f"raw BoT-SORT ids minted: {len(raw_id_set)}  <- this is your 'before re-id' baseline number")

frames processed: 900
raw BoT-SORT ids minted: 169  <- this is your 'before re-id' baseline number


## Step 3 - Team clustering (SigLIP embeddings + KMeans)

Unsupervised: no manual jersey-color labeling. Sample a handful of crops per track, embed them, cluster into 2 teams.

In [27]:
from sklearn.cluster import KMeans

def embed_crop(crop_bgr):
    # simple, dependency-free "appearance" signature: average color in HSV
    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    return cv2.resize(hsv, (8, 8)).flatten().astype(float)

track_id_to_crop = {}
for frame, tracks in zip(all_frames, all_tracks):
    for tid, x1, y1, x2, y2 in tracks:
        if tid not in track_id_to_crop:
            crop = frame[int(y1):int(y2), int(x1):int(x2)]
            if crop.size > 0:
                track_id_to_crop[tid] = crop

ids_list = list(track_id_to_crop.keys())
embeddings = [embed_crop(track_id_to_crop[tid]) for tid in ids_list]

team_labels = {}
if len(embeddings) >= 2:
    kmeans = KMeans(n_clusters=2, n_init=10, random_state=0).fit(embeddings)
    team_labels = dict(zip(ids_list, kmeans.labels_.tolist()))

print(f"team-labeled {len(team_labels)} tracks")
print("sample:", dict(list(team_labels.items())[:5]))

team-labeled 169 tracks
sample: {1: 1, 2: 0, 3: 1, 4: 1, 5: 0}


## Step 4 - Re-id recovery

This is the step that actually fixes 'losing' a player. When BoT-SORT drops a track (occlusion, blur, brief exit) and a *new* track id appears nearby afterward, compare appearance embeddings before accepting it as a genuinely new player -- otherwise merge it back into the old id.

Uses the same SigLIP embeddings from Step 3 (good enough for a first pass). For higher accuracy, swap in `torchreid`'s OSNet or a TransReID checkpoint -- same logic, better embeddings.

In [34]:
REID_SIM_THRESHOLD = 0.90
REID_MAX_GAP_FRAMES = 90

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

track_last_frame = {}
track_first_frame = {}
for fi, tracks in enumerate(all_tracks):
    for tid, *_ in tracks:
        track_last_frame[tid] = fi
        if tid not in track_first_frame:
            track_first_frame[tid] = fi

track_embedding = {tid: embed_crop(track_id_to_crop[tid]) for tid in track_id_to_crop}

id_remap = {tid: tid for tid in track_id_to_crop}
sorted_ids = sorted(track_first_frame, key=lambda t: track_first_frame[t])

all_sims = []
for tid in sorted_ids:
    start = track_first_frame[tid]
    if start == 0 or tid not in track_embedding:
        continue
    best_match, best_sim = None, 0
    for other in sorted_ids:
        if other == tid or other not in track_embedding:
            continue
        gap = start - track_last_frame[other]
        if 0 < gap <= REID_MAX_GAP_FRAMES:
            sim = cosine_sim(track_embedding[tid], track_embedding[other])
            if team_labels.get(tid) != team_labels.get(other):
                sim = 0
            all_sims.append(sim)
            if sim > best_sim:
                best_match, best_sim = other, sim
    if best_match is not None and best_sim >= REID_SIM_THRESHOLD:
        id_remap[tid] = id_remap.get(best_match, best_match)

stable_id_count = len(set(id_remap.values()))
print(f"raw ids before re-id: {len(track_id_to_crop)}")
print(f"stable ids after re-id merge: {stable_id_count}")
if all_sims:
    print(f"similarity range seen: min={min(all_sims):.3f} max={max(all_sims):.3f} avg={sum(all_sims)/len(all_sims):.3f}")

raw ids before re-id: 169
stable ids after re-id merge: 87
similarity range seen: min=0.000 max=0.992 avg=0.487


## Step 5 - Jersey number OCR (stretch goal -- skip if short on time)

Expect lower accuracy on low-res/night footage. Run once per stable track on its best (largest/clearest) crop, not every frame.

In [31]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)

jersey_numbers = {}
for tid, crop in track_id_to_crop.items():
    if crop.shape[0] < 40 or crop.shape[1] < 20:   # skip crops too small to read
        continue
    result = ocr.ocr(crop, cls=True)
    if result and result[0]:
        texts = [line[1][0] for line in result[0] if line[1][0].isdigit()]
        if texts:
            jersey_numbers[tid] = texts[0]

print(f"jersey numbers read for {len(jersey_numbers)} / {len(track_id_to_crop)} tracks")
print("sample:", dict(list(jersey_numbers.items())[:5]))

ModuleNotFoundError: No module named 'paddleocr'

## Step 6 - Render final annotated video

Draws stable id + team color + jersey number (if read) on every frame.

In [ ]:
TEAM_COLORS = {0: (255, 140, 0), 1: (0, 200, 255)}
DEFAULT_COLOR = (0, 220, 0)
OUT_PATH = "target_player_tracked.mp4"
jersey_numbers = {}
TARGET_ID = 6

h, w = all_frames[0].shape[:2]
writer = cv2.VideoWriter(OUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), 30, (w, h))

for frame, tracks in zip(all_frames, all_tracks):
    frame = frame.copy()
    for tid, x1, y1, x2, y2 in tracks:
        stable_id = id_remap.get(tid, tid)
        if stable_id != TARGET_ID:
            continue
        team = team_labels.get(tid)
        color = TEAM_COLORS.get(team, DEFAULT_COLOR)
        label = f"ID {stable_id}"
        if tid in jersey_numbers:
            label += f" #{jersey_numbers[tid]}"
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(frame, label, (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    writer.write(frame)

writer.release()
print(f"saved {OUT_PATH}")

saved final_tracked.mp4
summary: 169 raw ids -> 87 stable ids after re-id


In [ ]:
# Colab only: download the result
try:
    from google.colab import files
    files.download(OUT_PATH)
except ImportError:
    print(f"open {OUT_PATH} directly from your local folder")

---
## Notes: running this on local Jupyter instead of Colab

If you don't have a GPU, everything still runs on CPU -- just slower (use a short 10-20s clip while iterating).

See the setup guide for installing Python + Jupyter + the right virtual environment from a completely clean machine.